# 11 — Graph RAG for Financial Documents

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Understand the difference between text RAG and Graph RAG.
2. Combine graph neighbours + text chunks as context for the LLM.
3. See a question where Graph RAG **beats** plain RAG.


## Text RAG vs Graph RAG

| Property | Text RAG | Graph RAG |
|---|---|---|
| Best for | "What does the policy say?" | "Who is connected to whom?" |
| Retrieval unit | Chunks of text | Nodes + edges (paths) |
| Strength | Free-form documents | Relationship-heavy data |
| Weakness | Multi-hop reasoning is hard | Building/curating the graph is work |


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


In [ ]:
from src.graph_utils import (
    build_finance_graph, find_related_party_paths,
    find_high_value_approvers, find_invoices_for_vendor,
)
from src.rag_utils import build_store_from_folder, rag_answer
from src.llm_client import ask_llm

G = build_finance_graph()
store = build_store_from_folder('data/generated/pdf')
print('Graph nodes:', len(G), '| Text store chunks:', len(store))

## 11.1 — Plain RAG on a relationship question

In [ ]:
q = ('List every related-party vendor, the invoices they issued, and which employee approved each invoice.')
print('--- TEXT RAG ---')
print(rag_answer(q, store, k=6))

Plain RAG can quote the *policy* on related parties and the *RPT listing* from the PDF, but it has **no idea who approved each invoice** because that comes from the structured ledger / approval workflow, not the text. Watch how Graph RAG fixes this.

## 11.2 — Graph RAG

We extract the relationships from the graph, format them as text, *then* let the LLM phrase the answer.

In [ ]:
def graph_context_for(question):
    # Naive router: pick a graph query based on keywords in the question.
    q = question.lower()
    if 'related party' in q:
        rows = find_related_party_paths(G)
        lines = ['Related-party paths from the graph:']
        for r in rows:
            lines.append(
                f"  - vendor {r['vendor']} ({r['vendor_name']}) issued {r['invoice']} "
                f"for NPR {r['amount']:,.0f}, approved by {r['approver']} ({r['approver_name']})."
            )
        return '\n'.join(lines)
    if 'high value' in q or 'above' in q:
        rows = find_high_value_approvers(G)
        return 'High-value approvers:\n' + '\n'.join(
            f"  - {r['invoice']} NPR {r['amount']:,.0f} by {r['approver_name']}" for r in rows
        )
    return ''

def graph_rag_answer(question, k=4):
    # 1) gather graph context
    g_ctx = graph_context_for(question)
    # 2) gather text context
    hits = store.search(question, k=k)
    t_ctx = '\n\n'.join(
        f"[{h['metadata'].get('source')} p.{h['metadata'].get('page','-')}]\n{h['text']}" for h in hits
    )
    # 3) ask the LLM to combine them
    prompt = (
        f'Question: {question}\n\n'
        f'GRAPH FACTS:\n{g_ctx}\n\n'
        f'DOCUMENT CONTEXT:\n{t_ctx}\n\n'
        'Answer the question using the graph facts and the document context. '
        'Cite documents by [source, page]. Treat the graph facts as ground-truth for who-approved-what.'
    )
    return ask_llm(prompt, system='You are a careful CA assistant. Use only the provided context.', temperature=0.1)

print('--- GRAPH RAG ---')
print(graph_rag_answer(q))

## 11.3 — Try more questions

In [ ]:
for q in [
    'Which related-party invoices were approved by the CFO?',
    'Which high-value (above NPR 5,00,000) approvals involve a related party?',
    'Summarise the loan covenants and any documents that disclose them.',
]:
    print('Q:', q)
    print(graph_rag_answer(q))
    print('-' * 80)

## Expected output

* Plain RAG answer in 11.1 *describes* the related-party rules but can't list specific approver-per-invoice.
* Graph RAG answer in 11.2 enumerates each path: vendor → invoice → amount → approver.


## Exercise

1. Extend `graph_context_for` to recognise a question about *covenants* and include the loan-covenant subgraph.
2. Add a new node-type **Department** and connect employees; ask *"which departments handle high-value approvals?"*


## Common errors

| Symptom | Fix |
|---|---|
| `graph_context_for` returns empty | Add a new keyword route, or include a fallback summary. |
| LLM ignores graph facts | Reword the prompt: *"Treat graph facts as ground-truth; do not contradict them."* |


## ⚠️ Professional caution

Graph RAG is only as good as the graph you built. A wrong edge becomes a wrong answer with high confidence. For real engagements, build the graph from authoritative sources only (the ERP, signed approvals, official registries).